# `alt_transformer_train` -- Training Loop for the Alternative Transformer

**Ported/adapted from `project_hrishav/train.py`**, kept deliberately small: standard supervised training loop with early stopping on validation Brier score, nothing more elaborate than that. This is appropriate for the SECONDARY/alternative model in this project -- it doesn't need the full seed-ablation/GNN-pretraining infrastructure hrishav's original project had, since this port's job is just to demonstrate the architecture works on gagan's data, not to reproduce hrishav's exact headline numbers (which were computed on different data entirely).

In [1]:
import numpy as np
import torch
from sklearn.metrics import brier_score_loss, roc_auc_score
from torch.utils.data import DataLoader

from src.transformer_model import IPLTransformer, MultiTaskLoss, InningsDataset, MAX_SEQ_LEN

SEED = 42

## `_final_ball_predictions()` -- evaluation helper

A private helper used both during training (to pick the best checkpoint by validation Brier) and for the final test-set report: runs the model in `eval()` mode (dropout disabled -- this is a *point estimate*, not the MC-Dropout uncertainty estimate that `win_probability_engine.py` provides), and for each innings in the dataset extracts the win-probability prediction at the **last real (non-padded) ball** -- i.e. "what did the model think right at the end of the innings?", which is the natural point to compare against the actual win/loss outcome.

`valid[i].sum().item() - 1` finds the index of the last real ball for sequence `i` (since `valid_mask` is a boolean array with all the real balls first, followed by padding, the count of `True` values minus 1 is exactly the last real index).

In [2]:
def _final_ball_predictions(model: IPLTransformer, dataset: InningsDataset, device: str) -> tuple[np.ndarray, np.ndarray]:
    """Win-probability prediction at the last real ball of each innings."""
    model.eval()
    loader = DataLoader(dataset, batch_size=32)
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in loader:
            x = batch["features"].to(device)
            valid = batch["valid_mask"]
            preds = model(x)["win_prob"].squeeze(-1).cpu().numpy()  # (B, T)
            for i in range(x.shape[0]):
                last_idx = int(valid[i].sum().item()) - 1
                if last_idx < 0:
                    continue
                y_pred.append(preds[i, last_idx])
                y_true.append(batch["win_labels"][i, 0].item())
    return np.array(y_true), np.array(y_pred)

## `train_alt_transformer()` -- the training loop

**Steps:**
1. Wrap the three sequence lists (train/val/test) in    `InningsDataset` objects.
2. Build a fresh `IPLTransformer` and `MultiTaskLoss` (with    whatever lambdas the caller passed -- defaults to win-only,    per `transformer_model.py`'s module-level defaults).
3. Use `AdamW` (Adam with decoupled weight decay -- generally    preferred over plain Adam for Transformers) as the    optimizer, `weight_decay=1e-4` for a small amount of    regularisation.
4. Standard training loop: for each epoch, iterate over shuffled    training batches, compute the multi-task loss, backpropagate,    **clip gradients to a max norm of 1.0** (a common stability    measure for Transformers, which can otherwise occasionally    produce large gradient spikes), and step the optimizer.
5. **After every epoch**, evaluate on the validation set and    keep a copy of the model's weights (`best_state`) whenever    validation Brier improves -- simple early-stopping-by-   checkpoint, avoiding overfitting to the specific epoch    count.
6. After all epochs, restore the best checkpoint (not    necessarily the last epoch's weights) and report final    test-set Brier/AUC at the last ball of each test innings.

The test-set metrics are only computed `if len(np.unique(y_test)) > 1` -- a defensive guard against the (edge-case) situation where a test split happens to contain only wins or only losses, in which case AUC is mathematically undefined.

In [3]:
def train_alt_transformer(
    train_seqs: list[dict], val_seqs: list[dict], test_seqs: list[dict],
    epochs: int = 8, batch_size: int = 32, lr: float = 1e-3,
    lambda_next_ball: float = 0.0, lambda_score: float = 0.0,
    device: str = "cpu",
) -> dict:
    """
    Train IPLTransformer win-only (default) or multi-task (if lambdas > 0)
    on the given train/val/test innings sequences. Returns the trained
    model plus test-set Brier/AUC at the final ball of each innings.
    """
    torch.manual_seed(SEED)

    train_ds = InningsDataset(train_seqs, max_len=MAX_SEQ_LEN)
    val_ds = InningsDataset(val_seqs, max_len=MAX_SEQ_LEN)
    test_ds = InningsDataset(test_seqs, max_len=MAX_SEQ_LEN)

    model = IPLTransformer().to(device)
    loss_fn = MultiTaskLoss(lambda_next_ball=lambda_next_ball, lambda_score=lambda_score)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    best_val_brier = float("inf")
    best_state = None

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            x = batch["features"].to(device)
            win_labels = batch["win_labels"].to(device)
            nb_labels = batch["nb_labels"].to(device)
            score_labels = batch["score_labels"].to(device)
            valid_mask = batch["valid_mask"].to(device)

            optimizer.zero_grad()
            preds = model(x)
            losses = loss_fn(preds, win_labels, nb_labels, score_labels, valid_mask)
            losses["loss"].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        y_val, p_val = _final_ball_predictions(model, val_ds, device)
        if len(y_val) > 0:
            val_brier = brier_score_loss(y_val, p_val)
            if val_brier < best_val_brier:
                best_val_brier = val_brier
                best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    y_test, p_test = _final_ball_predictions(model, test_ds, device)
    result = {"model": model, "best_val_brier": best_val_brier}
    if len(y_test) > 0 and len(np.unique(y_test)) > 1:
        result["test_brier"] = brier_score_loss(y_test, p_test)
        result["test_auc"] = roc_auc_score(y_test, p_test)
        result["test_n"] = len(y_test)
    return result